# Reduced states on a Gaussian square lattice

A `size x size` square lattice of `GfPEPS` nodes from `FreeFermion/GfPEPS.py`: one node per
site, carrying `ext_dim` external (physical) Majorana modes and one `bond_dim`-Majorana bond
to its right and down neighbour, with the covariance drawn from `randcov`, so every node
starts **pure**.

Two operations are easy to confuse, and they behave differently:

| operation | what it does to the modes | purity |
|---|---|---|
| contract a bond | inner product with the fermionic EPR state on the bond (`contract_sites`) | pure $\to$ pure |
| trace out a subsystem | keep the sub-block of the covariance (partial trace) | pure $\to$ mixed |

So *contract everything, then trace* is exact, while *cut the bonds first, then contract* is
the tensor-network approximation.


## 1. What the module provides

The mode bookkeeping lives in `GfPEPS` now:

* `state.contract_all()` — contract every bond into one node and return
  `(state, labels)`, where `labels[(node, mode)]` is the mode index in the result;
* `state.partial_trace(sites)` — the exact reduced state of a set of nodes (contract
  everything, then trace out the rest);
* `GfPEPS.from_covariances` / `from_global_covariance` / `from_hamiltonian` /
  `product_state` — public constructors instead of the private `_from_data`;
* `state[node]` — the covariance of a node as blocks keyed by leg (`state[node][a, b]`),
  so a virtual bond and the matrix entries that belong to it cannot drift apart;
* `purity`, `entropy`, `occupations`, `is_pure` (`FreeFermion/linalg.py`) — diagnostics of a covariance.

Only two helpers are local to this notebook: the lattice builder, and
`cut_patch_covariance`, the *cut the bonds first* approximation that the module deliberately
does not provide (it is not the exact reduced state).


In [1]:
import os
import sys

# the notebook lives in playground/, the repo root is its parent
sys.path.insert(0, os.path.abspath('..'))


In [2]:
import torch
from torch import Tensor

from base import CUDA
from FreeFermion.GfPEPS import Bond, Graph, GfPEPS
from FreeFermion.linalg import entropy, is_pure, occupations, purity


def square_lattice(size: int, ext_dim: int = 2, bond_dim: int = 2,
                   seed: int | None = None) -> GfPEPS:
    '''
    open square lattice of size x size nodes, one node per site, a bond to the
    right and to the down neighbour, and a random pure covariance on every node.
    Args:
        size: number of rows and columns.
        ext_dim: external Majorana modes per site.
        bond_dim: Majorana modes per bond.
        seed: optional torch seed for the random covariances.
    Returns:
        The GfPEPS with size**2 nodes.
    '''
    if seed is not None:
        torch.manual_seed(seed)
    edges = []
    for row in range(size):
        for column in range(size):
            node = row * size + column
            if column + 1 < size:
                edges.append(Bond(node, node + 1, bond_dim))
            if row + 1 < size:
                edges.append(Bond(node, node + size, bond_dim))
    return GfPEPS(Graph(size * size, tuple(edges)), [ext_dim] * (size * size))


def cut_patch_covariance(state: GfPEPS, sites: list[int]) -> Tensor:
    '''
    the tensor-network approximation: cut every bond that leaves the patch before
    contracting, i.e. keep on each patch node only its external modes and the bond
    blocks that stay inside, and then contract the patch alone.  Selecting the legs
    of a node is enough, because its blocks are keyed by leg: dropping a bond drops
    its blocks and the entries of the matrix together.
    Args:
        state: the Gaussian network.
        sites: the original node labels of the patch.
    Returns:
        The covariance of the patch, square of size 2 * ext_dim * len(sites).
    '''
    inside = sorted(sites)
    bonds = [bond for bond in state.graph.edges
             if bond.first in inside and bond.second in inside]
    tensors, ext_dim = [], []
    for node in inside:
        legs = [None] + [bond for bond in state.connected_nodes(node) if bond in bonds]
        tensors.append(state[node].assemble(legs))
        ext_dim.append(state.ext_dim[node])
    labels = {node: index for index, node in enumerate(inside)}
    edges = tuple(Bond(labels[bond.first], labels[bond.second], bond.bond_dim)
                  for bond in bonds)
    return GfPEPS.from_covariances(Graph(len(inside), edges), ext_dim,
                                   tensors).partial_trace(list(range(len(inside))))


## 2. Contracting the bonds keeps the state pure

In [3]:
size = 3
sites = [row * size + column for row in range(size) for column in range(size)]
lattice = square_lattice(size=size, ext_dim=2, bond_dim=2, seed=0)
print(f'{size}x{size} lattice: {len(sites)} nodes, {len(lattice.graph.edges)} bonds, '
      f'{sum(lattice.ext_dim)} external Majorana modes')
print('each node is pure          : is_pure = %s, purity = %.15f'
      % (is_pure(lattice.tensors[0]), purity(lattice.tensors[0])))

contracted, labels = lattice.contract_all()
print('all bonds contracted       : %s, is_pure = %s  <- still pure'
      % (tuple(contracted.tensors[0].shape), is_pure(contracted.tensors[0])))
print('  labels[(0, 1)] =', labels[(0, 1)], ' labels[(8, 1)] =', labels[(8, 1)])

# the order of the internal bond contractions does not matter
backwards, labels_backwards = lattice.contract_all(from_end=True)
print('contracting back to front  : is_pure = %s (its labels record the new mode order)'
      % is_pure(backwards.tensors[0]))
print('whole lattice, both orders : |diff| = %.1e'
      % float((lattice.partial_trace(sites) - lattice.partial_trace(sites, from_end=True)).abs().max()))


3x3 lattice: 9 nodes, 12 bonds, 18 external Majorana modes
each node is pure          : is_pure = True, purity = 0.999999999999999
all bonds contracted       : (18, 18), is_pure = True  <- still pure
  labels[(0, 1)] = 1  labels[(8, 1)] = 17
contracting back to front  : is_pure = True (its labels record the new mode order)
whole lattice, both orders : |diff| = 4.2e-16


## 3. Tracing out the environment makes the patch mixed

`partial_trace` contracts **all** bonds first (pure), then keeps the sub-block of the patch,
i.e. traces out the rest. The naive route cuts the bonds that leave the patch before
contracting, and differs.


In [4]:
for patch, name in [([4], 'centre site'), ([3, 4, 6, 7], '2x2 block'),
                     ([1, 3, 4, 5, 7], 'centre + neighbours')]:
    exact = lattice.partial_trace(patch)
    from_the_end = lattice.partial_trace(patch, from_end=True)
    naive = cut_patch_covariance(lattice, patch)
    print(f'patch {name:22s} ({len(patch)} sites, {exact.shape[0]:2d} modes)')
    print('  exact (contract all, then trace)  : is_pure = %s, purity = %.4f, entropy = %.4f'
          % (is_pure(exact), purity(exact), entropy(exact)))
    print('    occupations = %s' % [round(float(x), 5) for x in occupations(exact)])
    print('    both contraction orders agree   : |diff| = %.1e'
          % float((exact - from_the_end).abs().max()))
    print('  naive (cut the bonds, then contract): purity = %.4f, |diff vs exact| = %.4f'
          % (purity(naive), float((exact - naive).abs().max())))


patch centre site            (1 sites,  2 modes)
  exact (contract all, then trace)  : is_pure = False, purity = 0.5001, entropy = 0.6931
    occupations = [0.49334]
    both contraction orders agree   : |diff| = 6.5e-17
  naive (cut the bonds, then contract): purity = 0.5087, |diff vs exact| = 0.1454
patch 2x2 block              (4 sites,  8 modes)
  exact (contract all, then trace)  : is_pure = False, purity = 0.3443, entropy = 1.2509
    occupations = [0.45728, 0.1752, 0.01956, 0.0001]
    both contraction orders agree   : |diff| = 3.3e-16
  naive (cut the bonds, then contract): purity = 0.2911, |diff vs exact| = 0.2493
patch centre + neighbours    (5 sites, 10 modes)
  exact (contract all, then trace)  : is_pure = False, purity = 0.2684, entropy = 1.6597
    occupations = [0.37075, 0.23549, 0.06697, 0.05352, 0.0]
    both contraction orders agree   : |diff| = 2.8e-16
  naive (cut the bonds, then contract): purity = 0.0701, |diff vs exact| = 0.5933


## 4. Growing the patch back to the whole lattice

The mixedness and the entropy come from the modes that are traced out, so they shrink as the
patch grows and vanish when the patch is the whole lattice (nothing is traced out).


In [5]:
shapes = [([4], '1 site'), ([3, 4, 6, 7], '2x2 block'),
          ([1, 3, 4, 5, 7], '5 sites'), (sites, 'all 9 sites')]
print(f'{"patch":>12} {"modes":>6} {"purity":>9} {"entropy":>9}   occupations')
for patch, name in shapes:
    exact = lattice.partial_trace(patch)
    print(f'{name:>12} {exact.shape[0]:>6} {purity(exact):>9.4f} {entropy(exact):>9.4f}   '
          f'{[round(float(x), 4) for x in occupations(exact)]}')


       patch  modes    purity   entropy   occupations
      1 site      2    0.5001    0.6931   [0.4933]
   2x2 block      8    0.3443    1.2509   [0.4573, 0.1752, 0.0196, 0.0001]
     5 sites     10    0.2684    1.6597   [0.3707, 0.2355, 0.067, 0.0535, 0.0]
 all 9 sites     18    1.0000    0.0000   [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -0.0, -0.0, -0.0]
